## Silver — `cno` (Cadastro Nacional de Obras)

**Origem:** `workspace.bronze.cno` → **Destino:** `workspace.silver.cno`

- **Grão:** 1 linha por obra — em caso de duplicidade de `cno`, mantém o registro com `data_da_situacao` mais recente.
- **Transformações:**
  - Seleção apenas das colunas relevantes à análise: `cno`, `data_de_inicio`, `codigo_do_municipio`, `unidade_de_medida`, `area_total`, `situacao`, `data_da_situacao`.
  - Correção de tipos: datas convertidas para `date` (`yyyy-MM-dd`) e `area_total` para `double`; registros que não convertem viram nulos e são descartados.
  - Filtros de qualidade (cada regra contabilizada no relatório abaixo):
    - `cno` não nulo/vazio;
    - `data_de_inicio` e `data_da_situacao` válidas;
    - `codigo_do_municipio` não nulo, numérico e **com registro em** `workspace.silver.municipios.codigo_tom` — o CNO informa o código TOM (SIAFI), normalizado com pad de 4 dígitos (`'33'` → `'0033'`) para casar com o enriquecimento da dimensão de municípios;
    - `unidade_de_medida` = `m²` (apenas metros quadrados; `km` e `,m2` são descartados);
    - `area_total` não nula e >= 0;
    - `situacao` dentro do domínio oficial RFB (01, 02, 03, 14, 15).
- **Relatório de qualidade:** quantidade de inválidos por regra + % sobre o total bronze.
- **Diagnóstico auxiliar:** taxa de adesão comparando o código informado como TOM (4 dígitos) e como IBGE (7 dígitos) — evidencia o padrão realmente usado no arquivo.
- **Linhagem:** CSV dados.gov.br → `bronze.cno` → limpeza/validação (TOM vs `silver.municipios.codigo_tom`, enriquecido via cidade-ibge-tom/SIAFI) → `silver.cno`.

In [0]:
%run ../shared/_setup

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F
from data_pipeline import (
    save_table,
    add_column_comments,
    add_table_comment,
    resumo_invalidos,
    condicao_valida,
    para_data_segura,
    para_double_seguro,
)
from catalogo.cadastro_nacional_obras import (
    SILVER_CNO_COMMENTS,
    SILVER_CNO_TABLE_COMMENT,
    DOMINIO_SITUACAO,
)

In [0]:
SOURCE_TABLE = "workspace.bronze.cno"
TARGET_TABLE = "workspace.silver.cno"
MUNICIPIOS_TABLE = "workspace.silver.municipios"

# Contrato de saída silver.cno
COLUNAS_FINAIS = [
    "cno",
    "data_de_inicio",
    "codigo_do_municipio",
    "unidade_de_medida",
    "area_total",
    "situacao",
    "data_da_situacao",
]

In [0]:
df_bronze = spark.table(SOURCE_TABLE)
total_bronze = df_bronze.count()
print(f"Bronze: {total_bronze:,} linhas | colunas: {df_bronze.columns}")

# Seleção das colunas relevantes com conversão tolerante
# (para_data_segura/para_double_seguro): valor malformado vira null e é contabilizado
# no relatório de qualidade, sem erro de cast mesmo com ANSI mode habilitado
df = df_bronze.select(
    F.trim(F.col("cno").cast("string")).alias("cno"),
    para_data_segura(F.col("data_de_inicio")).alias("data_de_inicio"),
    F.trim(F.col("codigo_do_municipio").cast("string")).alias("codigo_do_municipio"),
    F.trim(F.col("unidade_de_medida").cast("string")).alias("unidade_de_medida"),
    para_double_seguro(F.col("area_total")).alias("area_total"),
    F.trim(F.col("situacao").cast("string")).alias("situacao"),
    para_data_segura(F.col("data_da_situacao")).alias("data_da_situacao"),
)
display(df.limit(5))

In [0]:
# Validação do código do município: o CNO informa o código TOM (SIAFI),
# comparado com o codigo_tom enriquecido em silver.municipios (pad 4 dígitos)
df_municipios = (
    spark.table(MUNICIPIOS_TABLE)
    .select(F.lpad(F.col("codigo_tom"), 4, "0").alias("tom_pad"))
    .where(F.col("tom_pad").isNotNull())
    .distinct()
    .withColumn("municipio_valido", F.lit(True))
)

df = df.withColumn("_tom", F.when(F.col("codigo_do_municipio").rlike("^[0-9]{1,4}$"), F.lpad(F.col("codigo_do_municipio"), 4, "0")))
df = df.join(df_municipios, df["_tom"] == df_municipios["tom_pad"], how="left")

# Diagnóstico: qual padrão o arquivo realmente usa? (TOM x IBGE)
toms_distintos = spark.table(MUNICIPIOS_TABLE).select("codigo_tom").distinct()
ibges_distintos = spark.table(MUNICIPIOS_TABLE).select("codigo_municipio").distinct()
match_tom = df.filter(F.col("municipio_valido")).count()
match_ibge = (
    df.join(ibges_distintos, df["codigo_do_municipio"] == ibges_distintos["codigo_municipio"], how="inner")
    .count()
)
print(f"Adesão como TOM (pad 4): {match_tom:,} | Adesão como IBGE (7 díg): {match_ibge:,}")
if match_tom < match_ibge:
    print("ALERTA: adesão maior como IBGE — avaliar validar por codigo_municipio em vez de TOM.")

# Normaliza o código para o formato canônico TOM (pad de 4 dígitos)
df = df.drop("tom_pad").withColumn(
    "codigo_do_municipio",
    F.coalesce(F.col("_tom"), F.col("codigo_do_municipio")),
).drop("_tom")

In [0]:
REGRAS_INVALIDOS = {
    "cno_nulo_ou_vazio": F.col("cno").isNull() | (F.col("cno") == ""),
    "data_de_inicio_nula_ou_invalida": F.col("data_de_inicio").isNull(),
    "codigo_do_municipio_nulo_ou_nao_numerico": (
        F.col("codigo_do_municipio").isNull()
        | ~F.col("codigo_do_municipio").rlike("^[0-9]{1,7}$")
    ),
    "codigo_do_municipio_sem_registro_em_municipios": ~F.coalesce(F.col("municipio_valido"), F.lit(False)),
    "unidade_de_medida_inválida":  (F.col("unidade_de_medida") != "m2"),
    "area_total_nula_ou_negativa": F.col("area_total").isNull() | (F.col("area_total") < 0),
    "situacao_fora_do_dominio": ~F.coalesce(F.col("situacao").isin(list(DOMINIO_SITUACAO)), F.lit(False)),
    "data_da_situacao_nula_ou_invalida": F.col("data_da_situacao").isNull(),
}

In [0]:
df_relatorio = resumo_invalidos(spark, df, REGRAS_INVALIDOS)
print(f"Total bronze avaliado: {total_bronze:,}")
display(df_relatorio)

In [0]:
# Mantém apenas registros válidos em todas as regras
df = df.filter(condicao_valida(REGRAS_INVALIDOS))

# Deduplicação determinística: mantém a situação mais recente da obra
w = Window.partitionBy("cno").orderBy(F.col("data_da_situacao").desc())
antes_dedup = df.count()
df = (
    df.withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .drop("_rn", "municipio_valido")
)
print(f"Válidos: {antes_dedup:,} | Após dedup por cno: {df.count():,}")

df = df.select(*COLUNAS_FINAIS)
display(df.limit(10))

In [0]:
save_table(df, TARGET_TABLE)
add_column_comments(
    spark,
    TARGET_TABLE,
    SILVER_CNO_COMMENTS
)
add_table_comment(spark, TARGET_TABLE, SILVER_CNO_TABLE_COMMENT)
print(f"Tabela {TARGET_TABLE} persistida: {spark.table(TARGET_TABLE).count():,} linhas")

In [0]:
total = spark.table(TARGET_TABLE).count()
distintos = spark.table(TARGET_TABLE).select("cno").distinct().count()
print(f"Total: {total:,} | CNO distintos: {distintos:,} | Duplicatas: {total - distintos:,}")
assert total == distintos, "Quebra de unicidade de cno em silver.cno"
display(spark.sql(f"SELECT situacao, count(*) AS qtd_obras FROM {TARGET_TABLE} GROUP BY situacao ORDER BY situacao"))
display(spark.sql(f"SELECT min(data_de_inicio) AS primeira_inicio, max(data_de_inicio) AS ultima_inicio FROM {TARGET_TABLE}"))
display(spark.sql(f"DESCRIBE TABLE {TARGET_TABLE}"))